# Tutorial 2 — Hugging Face QLoRA (4-bit NF4 + PEFT)

**Goal:** Fine-tune a pretrained DistilBERT model for IMDB sentiment classification using **4-bit NF4 base weights** and **trainable LoRA adapters**.

**Environment:** Run in **Google Colab with an NVIDIA CUDA GPU** (or another compatible CUDA environment). This notebook intentionally checks for CUDA and stops otherwise. The standard bitsandbytes 4-bit training path used here is not a general-purpose Apple MPS solution.

**What you will learn:** load model/tokenizer, inspect layer names, configure NF4/double quantization, prepare for k-bit training, attach PEFT adapters, tokenize IMDB, train, evaluate, inspect gradients/parameters, save/reload adapters, and run inference.

**Important:** The base checkpoint `distilbert/distilbert-base-uncased` is a general-language model, not an already trained sentiment classifier. Its classification head is randomly initialized and trained along with LoRA. The dataset and model require internet access on first download.

## Step 0 — Install packages

Run the installation cell, then **restart the runtime if the notebook requests it**. Compatible versions matter for 4-bit support.

In [ ]:
%pip install -q -U torch transformers peft accelerate bitsandbytes datasets scikit-learn

## Step 1 — Verify NVIDIA CUDA support

NF4 training requires a supported bitsandbytes backend. This tutorial targets CUDA, not MPS.

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Switch to a CUDA GPU runtime (e.g. Colab: Runtime > Change runtime type > T4 GPU).')
print('GPU:', torch.cuda.get_device_name(0))
print('Compute capability:', torch.cuda.get_device_capability(0))

## Step 2 — Load tokenizer and define 4-bit NF4 quantization

- `load_in_4bit=True`: quantize supported linear base weights.
- `nf4`: NormalFloat4 quantization type commonly used in QLoRA.
- `double_quant=True`: quantize some quantization constants to save additional memory.
- `compute_dtype=float16`: arithmetic precision for supported quantized operations; this is **not** the storage precision of base weights.

4-bit storage does **not** mean all model tensors use 4 bits; embeddings, layer norms, heads, metadata, activations, and adapters have other dtypes.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, BitsAndBytesConfig

MODEL_ID = 'distilbert/distilbert-base-uncased'
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print('Pad token:', tokenizer.pad_token)

## Step 3 — Load the pretrained model with a 2-class head

The classification head is newly initialized because the base checkpoint is not a sentiment classifier. The library constructs the model architecture automatically; you do not need to recreate its Python class blueprint.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    id2label={0:'NEGATIVE', 1:'POSITIVE'},
    label2id={'NEGATIVE':0, 'POSITIVE':1},
    quantization_config=quant_config,
    device_map={'': 0},
)
print('Model type:', type(model).__name__)
print('Base device:', next(model.parameters()).device)

## Step 4 — Inspect LoRA target modules

DistilBERT attention modules include `q_lin`, `k_lin`, `v_lin`, and `out_lin`. We select query and value projections. Names differ across model families (BERT often uses `query`/`value`, Llama uses `q_proj`/`v_proj`).

In [ ]:
for name, module in model.named_modules():
    if name.endswith(('q_lin', 'v_lin')):
        print(name, type(module).__name__)
print('Classifier:', type(model.classifier).__name__)

## Step 5 — Prepare k-bit model and add LoRA

`prepare_model_for_kbit_training` freezes base parameters and performs preparation useful for low-bit training. PEFT inserts LoRA adapters and keeps the task head trainable. `modules_to_save` explicitly preserves the DistilBERT `pre_classifier` and `classifier` layers.

**Scaling difference from your lab:** Your lab used `alpha * (x@A@B)`; PEFT ordinarily uses `(lora_alpha/r) * correction`. With `r=8`, `lora_alpha=16`, effective scaling is 2.

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)
config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=['q_lin', 'v_lin'],
    bias='none',
    task_type=TaskType.SEQ_CLS,
    modules_to_save=['pre_classifier', 'classifier'],
)
model = get_peft_model(model, config)
model.print_trainable_parameters()
for name, param in model.named_parameters():
    if param.requires_grad and ('lora_' in name or 'classifier' in name):
        print(name, tuple(param.shape))

## Step 6 — Load IMDB and tokenize

IMDB is a real binary sentiment dataset. To keep the tutorial practical on a modest Colab GPU, we train on 1,000 examples and evaluate on 200. Increase the sizes later for a meaningful benchmark.

In [ ]:
from datasets import load_dataset
raw = load_dataset('imdb')
train_ds = raw['train'].shuffle(seed=42).select(range(1000))
eval_ds = raw['test'].shuffle(seed=42).select(range(200))

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=256)

train_ds = train_ds.map(tokenize, batched=True, remove_columns=['text'])
eval_ds = eval_ds.map(tokenize, batched=True, remove_columns=['text'])
print(train_ds[0].keys())
print('Training samples:', len(train_ds), '| Evaluation samples:', len(eval_ds))

## Step 7 — Define evaluation metrics

Accuracy is easy to interpret, but for more serious experiments also consider F1, precision/recall, multiple random seeds, and class balance.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    predictions = np.argmax(logits, axis=-1)
    return {
        'accuracy': accuracy_score(labels, predictions),
        'f1': f1_score(labels, predictions),
    }

## Step 8 — Configure Trainer

Use mixed precision on CUDA and gradient accumulation to limit per-device batch size. This example uses the standard AdamW optimizer; QLoRA workflows can also use paged optimizers for larger models. `eval_strategy` is the argument used in recent Transformers versions.

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding

args = TrainingArguments(
    output_dir='./distilbert_imdb_qlora_outputs',
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=2,
    learning_rate=2e-4,
    fp16=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    logging_steps=20,
    report_to='none',
    remove_unused_columns=True,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics,
)
print('Ready to train')

## Step 9 — Train and evaluate

The base weights stay frozen and quantized; LoRA A/B and the classifier modules learn. GPU memory and speed depend on hardware, sequence length, and software versions.

In [ ]:
train_result = trainer.train()
print('Train metrics:', train_result.metrics)
metrics = trainer.evaluate()
print('Evaluation:', metrics)
if torch.cuda.is_available():
    print('Peak allocated GPU memory (GiB):', round(torch.cuda.max_memory_allocated()/1024**3, 3))

## Step 10 — Inspect trained adapters and base quantization

LoRA matrices in PEFT are generally named `lora_A` and `lora_B`, and their orientation follows `nn.Linear`'s weight storage convention. This differs from the row-vector A/B notation in your custom lab, but the underlying low-rank correction is equivalent.

In [ ]:
trainable = [(n, p.numel()) for n, p in model.named_parameters() if p.requires_grad]
print('Trainable parameter count:', sum(count for _, count in trainable))
for name, count in trainable[:20]:
    print(name, count)

# Inspect bitsandbytes quantized linear modules.
import bitsandbytes as bnb
quantized = [(n, m) for n, m in model.named_modules() if isinstance(m, bnb.nn.Linear4bit)]
print('Number of Linear4bit modules:', len(quantized))
print('Example quantized module:', quantized[0][0] if quantized else 'none')

## Step 11 — Run sentiment inference

The tokenizer produces input IDs and attention masks. `model.eval()` disables dropout; `torch.no_grad()` avoids tracking inference gradients.

In [ ]:
model.eval()
texts = [
    'This movie was wonderful, moving, and beautifully acted.',
    'The film was dull and disappointing.',
]
inputs = tokenizer(texts, return_tensors='pt', padding=True, truncation=True, max_length=256)
inputs = {k: v.to('cuda:0') for k, v in inputs.items()}
with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).float().cpu()
for text, p in zip(texts, probs):
    print(text, '=>', {model.config.id2label[i]: round(float(p[i]), 3) for i in range(2)})

## Step 12 — Save the adapter and reload it

`save_pretrained` on a PEFT model saves the adapter configuration/weights and configured task modules, **not** the full base model. Reloading requires the same base checkpoint and a compatible quantization configuration.

In [ ]:
ADAPTER_DIR = './imdb_distilbert_nf4_qlora_adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print('Saved PEFT adapter to:', ADAPTER_DIR)

# Optional reload in a fresh session:
# from peft import PeftModel
# base = AutoModelForSequenceClassification.from_pretrained(
#     MODEL_ID, num_labels=2, quantization_config=quant_config,
#     device_map={'': 0},
# )
# restored = PeftModel.from_pretrained(base, ADAPTER_DIR)
# restored.eval()

## Step 13 — Exercises and troubleshooting

1. Change rank to 4 or 16. How do the number of trainable A/B weights and GPU memory change?
2. Compare `target_modules=['q_lin','v_lin']` against `target_modules='all-linear'`.
3. Compare LoRA alpha values 8 and 16 at the same rank.
4. Increase IMDB sample count; compare accuracy, training time, and peak allocated GPU memory.
5. Compare against full-precision LoRA on the same dataset and hardware, holding other settings fixed.

**Troubleshooting:**
- **No CUDA:** Use a Colab GPU runtime. This example intentionally does not fall back to CPU/MPS.
- **Out of memory:** Reduce `max_length` from 256 to 128 and/or batch size from 8 to 4.
- **Missing packages:** Re-run installation and restart the runtime.
- **Unexpected target module names:** Print `model.named_modules()`; names differ by architecture.
- **Unexpected head warning:** DistilBERT's sequence-classification head is newly initialized by design.
- **Version incompatibilities:** Upgrade `transformers`, `peft`, `accelerate`, and `bitsandbytes` together.

**Summary:** Canonical QLoRA combines frozen 4-bit NF4 pretrained base weights with higher-precision trainable LoRA adapters. It lowers base-weight storage, but total memory also includes activations, adapters, optimizer states, and nonquantized layers.